# Lección 13.7 — Fine-tuning eficiente: LoRA y cuantización (intuición)

> **Módulo 13 · Modelos generativos y LLMs** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 3.3 (descenso de gradiente), Lección 12.5 (leyes de escalamiento), Lección 13.5 (cómo se entrena un LLM)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué ajustar (*fine-tuning*) un LLM completo es tan caro en memoria.
- Entender qué es una matriz de **bajo rango** y por qué **LoRA** entrena solo una fracción mínima de los parámetros.
- Entrenar un adaptador LoRA en PyTorch y compararlo con el ajuste completo.
- Cuantizar pesos a **int8** e **int4**, medir el error y su efecto en GPT-2.
- Elegir valores de partida para el rango `r`, `alpha`, los módulos objetivo y los bits.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez. Carga PyTorch y GPT-2 small (lo usaremos como "LLM de verdad" para contar parámetros y cuantizarlo).

In [ ]:
import copy, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)                      # semilla fija: así tus números coinciden con las respuestas
tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").eval()
plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: notas adhesivas y precios redondeados

**LoRA: no reescribas el libro, pégale notas.** Tienes una enciclopedia de 20 tomos (el modelo pre-entrenado) y quieres adaptarla a tu empresa. Reescribir cada página (ajuste completo) es carísimo. En cambio, pegas unas pocas **notas adhesivas** con las correcciones. El libro queda intacto; las notas pesan poco y puedes tener un juego distinto para cada cliente.

**Cuantización: redondear los precios.** Si en vez de anotar \$12.3871 anotas \$12, cada número ocupa menos espacio. Pierdes un poquito de precisión, pero casi siempre la cuenta final apenas cambia. Un LLM tiene **miles de millones** de números: redondearlos reduce mucho la memoria que necesita.

```
LoRA          → entrenar MENOS números       (memoria para entrenar ↓↓)
Cuantización  → guardar cada número en MENOS bits  (memoria para usar ↓↓)
QLoRA         → las dos cosas a la vez
```

### ✋ Ejercicio 1 — ¿LoRA o cuantización?
- a) Quieres que un modelo de 7,000 millones de parámetros corra en una laptop con poca memoria.
- b) Quieres que el mismo modelo aprenda el estilo de los correos de tu empresa, pero solo tienes una GPU modesta.
- c) Quieres 50 versiones del modelo, una por cliente, sin guardar 50 copias completas.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

a) **Cuantización**: mismo modelo, menos bits por número → menos memoria para usarlo.
b) **LoRA** (o QLoRA si ni así cabe): entrenas solo una fracción pequeñísima de parámetros.
c) **LoRA**: guardas una sola copia del modelo base y 50 juegos de "notas adhesivas" (adaptadores) de pocos megabytes cada uno.
</details>

---
## 2. 📖 El problema: ajustar un LLM completo no cabe en memoria

Para entrenar con Adam (Lección 3.3), por **cada** parámetro hay que guardar: el peso, su gradiente y dos "memorias" del optimizador. En 32 bits (4 bytes) son ≈ **16 bytes por parámetro**, sin contar las activaciones.

### 🔮 Ejercicio 2 — ¿Cuánta memoria?
GPT-2 small tiene 124 millones de parámetros. Un modelo "pequeño" de hoy tiene **7,000 millones**. ¿Cuántos GB harían falta para ajustarlo completo con Adam? Una GPU de Colab gratis tiene unos 15 GB.

✍️ *Mi estimación:*

In [ ]:
def memoria_gb(n_parametros, bytes_por_parametro):
    return n_parametros * bytes_por_parametro / 1e9

for nombre, n in [("GPT-2 small", 124e6), ("Modelo de 7B", 7e9), ("Modelo de 70B", 70e9)]:
    print(f"{nombre:<14} solo usarlo (32 bits): {memoria_gb(n, 4):7.1f} GB · "
          f"ajuste completo con Adam: {memoria_gb(n, 16):8.1f} GB")

<details><summary><b>👀 Ver respuesta</b></summary>

Un modelo de 7B necesita ≈ **112 GB** para el ajuste completo (y 28 GB solo para usarlo en 32 bits). No cabe ni de lejos en una GPU de 15 GB; hacen falta varias GPUs grandes. Con 70B, más de un terabyte. GPT-2 small (≈2 GB) sí cabe: por eso lo usamos aquí.

LoRA ataca el término "16 bytes por parámetro" (casi ningún parámetro necesita gradiente ni Adam) y la cuantización ataca el "4 bytes por peso".
</details>

---
## 3. 📖 Cómo funciona LoRA por dentro: matrices de bajo rango

Cada capa de un Transformer guarda sus pesos en una **matriz** `W` (una tabla de números) de tamaño, por ejemplo, 768 × 768.

La idea de LoRA (*Low-Rank Adaptation*): el **cambio** que necesita `W` para una tarea nueva suele ser "simple". En lugar de aprender una matriz completa `ΔW`, la escribimos como el producto de dos matrices **flacas**:

```
   ΔW  (d × d)     ≈      B (d × r)   ·   A (r × d)          con r muy pequeño (4, 8, 16)

 768 × 768 = 589,824 números      768×8 + 8×768 = 12,288 números   (¡48 veces menos!)
```

A ese `r` se le llama **rango**: cuántas "direcciones" distintas puede representar el cambio. Veámoslo primero a mano, con numpy.

In [ ]:
rng = np.random.default_rng(0)
# Una matriz 60×80 que en el fondo es "simple": producto de dos matrices flacas de rango 3, más un poco de ruido
W = rng.normal(size=(60, 3)) @ rng.normal(size=(3, 80)) + 0.1 * rng.normal(size=(60, 80))

valores_singulares = np.linalg.svd(W, compute_uv=False)
print("Los 8 valores singulares más grandes:", np.round(valores_singulares[:8], 1))

### 🔮 Ejercicio 3 — ¿Cuántas direcciones importan?
La **SVD** (descomposición en valores singulares) separa la matriz en "direcciones" ordenadas por importancia; cada **valor singular** mide cuánto aporta una dirección.

Mirando los números de arriba: ¿con cuántas direcciones (rango `r`) crees que se puede reconstruir `W` casi perfectamente? Predice y luego ejecuta la celda de abajo.

✍️ *Mi predicción:* r = ___

In [ ]:
U, S, Vt = np.linalg.svd(W, full_matrices=False)
for r in [1, 2, 3, 5, 20]:      # ✏️ prueba otros rangos
    W_r = (U[:, :r] * S[:r]) @ Vt[:r]                     # reconstrucción con solo r direcciones
    error = np.abs(W - W_r).mean() / np.abs(W).mean()
    print(f"r={r:>2} → error relativo {error:6.1%} · números guardados {r * (60 + 80):>5} (de {60 * 80})")

<details><summary><b>👀 Ver respuesta</b></summary>

**r = 3.** Hay tres valores singulares grandes (≈87, 63 y 55) y después todos caen de golpe a ≈1.5: eso es solo el ruido. Con `r=3` el error relativo baja a ≈**6 %** guardando 420 números en lugar de 4,800 (¡11 veces menos!). Subir a `r=5` o `r=20` casi no mejora: solo empieza a copiar el ruido.

Esa es la apuesta de LoRA: **el cambio útil vive en pocas direcciones.**
</details>

### 📖 La capa LoRA en PyTorch
Ahora lo hacemos de verdad. Una capa lineal "pre-entrenada" (`base`) queda **congelada** y le sumamos un adaptador:

$$\text{salida} = W x + \frac{\alpha}{r}\, B A x$$

- `A` empieza con valores pequeños al azar y `B` con **ceros**: al inicio el adaptador no cambia nada y el modelo se comporta exactamente como el original.
- `alpha` (α) es un factor de escala: controla cuánto "pesa" el adaptador.

Nuestra tarea: la capa base sabe hacer la tarea vieja, y queremos adaptarla a una **tarea nueva** que difiere de la vieja en un cambio de rango 2 (como en la vida real, donde el cambio entre "texto general" y "texto de tu empresa" suele ser de bajo rango).

In [ ]:
class CapaLoRA(nn.Module):
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                                     # 🧊 la capa original queda congelada
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)  # matriz flaca 1
        self.B = nn.Parameter(torch.zeros(base.out_features, r))        # matriz flaca 2 (empieza en cero)
        self.escala = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.escala

# --- La tarea: una capa de 64 → 64 ---
torch.manual_seed(0)
d = 64
W_vieja = torch.randn(d, d) / 8                                   # lo que el modelo ya sabía
cambio = (torch.randn(d, 2) @ torch.randn(2, d)) / 8              # lo nuevo: un cambio de rango 2
X_prueba = torch.randn(500, d)
Y_prueba = X_prueba @ (W_vieja + cambio).T

def datos_entrenamiento(n):
    g = torch.Generator().manual_seed(1)
    X = torch.randn(n, d, generator=g)
    return X, X @ (W_vieja + cambio).T + 0.05 * torch.randn(n, d, generator=g)

base = nn.Linear(d, d, bias=False)
base.weight.data = W_vieja.clone()

def entrenar(modelo, X, Y, pasos=300, lr=1e-2):
    entrenables = [p for p in modelo.parameters() if p.requires_grad]
    opt = torch.optim.Adam(entrenables, lr=lr)
    for _ in range(pasos):
        perdida = ((modelo(X) - Y) ** 2).mean()
        opt.zero_grad(); perdida.backward(); opt.step()
    with torch.no_grad():
        return ((modelo(X_prueba) - Y_prueba) ** 2).mean().item()

with torch.no_grad():
    print(f"Error de la capa original en la tarea nueva: {((base(X_prueba) - Y_prueba) ** 2).mean():.3f}")

### 🔮 Ejercicio 4 — Contar parámetros
La capa tiene 64 × 64 = 4,096 pesos. ¿Cuántos parámetros entrenables tiene un adaptador LoRA con `r=2`? ¿Y con `r=16`?

✍️ *Mis cuentas:* r=2 → ___ · r=16 → ___

In [ ]:
N_EJEMPLOS = 500    # ✏️ (lo usaremos en el Ejercicio 6)
X_tr, Y_tr = datos_entrenamiento(N_EJEMPLOS)

for r in [1, 2, 4, 16]:
    torch.manual_seed(0)
    modelo = CapaLoRA(copy.deepcopy(base), r=r, alpha=2 * r)
    n_ent = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    print(f"LoRA r={r:>2}: {n_ent:>5} parámetros entrenables → error en prueba {entrenar(modelo, X_tr, Y_tr):.5f}")

torch.manual_seed(0)
completo = copy.deepcopy(base)                                      # ajuste completo: se entrena TODO
n_ent = sum(p.numel() for p in completo.parameters())
print(f"Ajuste completo: {n_ent:>5} parámetros entrenables → error en prueba {entrenar(completo, X_tr, Y_tr):.5f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `r=2`: 2·64 + 64·2 = **256** parámetros (el 6 % de la capa). `r=16`: 16·(64+64) = **2,048**.
- Resultados: con `r=2` el error en prueba cae de ≈1.87 a ≈**0.00002**, prácticamente perfecto, entrenando 16 veces menos parámetros que el ajuste completo.
- Curioso: el ajuste completo (4,096 parámetros) queda **peor** en prueba (≈0.0004). Tiene tanta libertad que también aprende el ruido de los datos de entrenamiento: un poquito de **sobreajuste** (Lección 4.4). `r=4` y `r=16` también funcionan, pero cada vez se acercan más a ese comportamiento.
</details>

### 🔮 Ejercicio 5 — ¿Y si `r` es demasiado pequeño?
Mira la fila `r=1` de la tabla anterior. El cambio real era de rango 2. ¿Por qué `r=1` falla aunque entrene igual de bien?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con `r=1` el error se queda en ≈**0.77**: mejora respecto al original (≈1.87), pero muy lejos de los demás. El cambio real necesita **dos** direcciones y el adaptador solo puede representar **una**: por mucho que entrene, le falta capacidad. Es **subajuste**. Por eso `r` es un hiperparámetro: demasiado bajo no alcanza; demasiado alto desperdicia memoria y se acerca al ajuste completo.
</details>

### ✏️ Ejercicio 6 — ¿Qué pasa si tengo pocos datos?
En la celda del Ejercicio 4 cambia `N_EJEMPLOS = 100` y vuelve a ejecutarla. Antes de hacerlo: ¿quién sufrirá más con pocos datos, LoRA `r=2` o el ajuste completo?

✍️ *Mi predicción:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con `N_EJEMPLOS = 100` el **ajuste completo sufre mucho más**: su error en prueba sube a ≈**0.006** (antes ≈0.0004, unas 15 veces peor), mientras que LoRA `r=2` queda en ≈**0.0002**, casi 30 veces mejor que el ajuste completo. Con pocos datos, 4,096 parámetros libres memorizan el ruido; 256 parámetros con la estructura correcta generalizan.

Por eso LoRA no solo es barato: con conjuntos de datos pequeños (lo normal al adaptar un LLM a una empresa) **suele generalizar mejor** y **olvidar menos** lo que el modelo ya sabía.
</details>

### 📖 LoRA en un LLM de verdad
En un Transformer, LoRA se aplica a algunas matrices de cada bloque (Lección 12.3): normalmente las de **atención** (consultas, claves, valores) y a veces también las del **MLP**. En GPT-2 se llaman:

| Módulo | Qué es | Tamaño en GPT-2 small |
|---|---|---|
| `attn.c_attn` | Q, K y V juntas | 768 × 2,304 |
| `attn.c_proj` | Salida de la atención | 768 × 768 |
| `mlp.c_fc` | Primera capa del MLP | 768 × 3,072 |
| `mlp.c_proj` | Segunda capa del MLP | 3,072 × 768 |

### 🔮 Ejercicio 7 — ¿Qué fracción de GPT-2 entrenaría LoRA?
Con `r=8` solo en `c_attn` de los 12 bloques, ¿qué porcentaje de los 124 millones de parámetros entrenarías? ¿Más o menos del 1 %?

✍️ *Mi predicción:*

In [ ]:
def parametros_lora(modelo, r, modulos):
    total = 0
    for nombre, capa in modelo.named_modules():
        if any(nombre.endswith(m) for m in modulos):
            entrada, salida = capa.weight.shape             # en GPT-2 los pesos se guardan como (entrada, salida)
            total += r * (entrada + salida)                 # A: r×entrada  +  B: salida×r
    return total

n_total = sum(p.numel() for p in gpt2.parameters())
for r, modulos in [(8, ["attn.c_attn"]),
                   (8, ["attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"]),
                   (64, ["attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"])]:   # ✏️
    n = parametros_lora(gpt2, r, modulos)
    print(f"r={r:<3} módulos={len(modulos)} → {n:>10,} entrenables = {n / n_total:.3%} del modelo "
          f"(adaptador de {n * 4 / 1e6:.1f} MB)")

<details><summary><b>👀 Ver respuesta</b></summary>

**Mucho menos del 1 %:** solo en `c_attn` con `r=8` son **294,912** parámetros, el **0.24 %** de GPT-2 (un adaptador de ≈1.2 MB). Poniéndolo en los cuatro módulos sube a ≈0.95 %; con `r=64` en los cuatro llega a ≈7.6 % (≈38 MB). En modelos de miles de millones de parámetros la fracción es todavía menor, típicamente 0.1 %.
</details>

### 📖 Hiperparámetros de LoRA

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `r` (rango) | Cuántas "direcciones" puede cambiar el adaptador | Más capacidad, más memoria, más riesgo de sobreajuste | Más barato; si es menor que lo que la tarea necesita, **subajuste** | 8–16 (4 si hay pocos datos, 64 para cambios grandes) |
| `alpha` (α) | Escala del adaptador (se usa α/r) | El adaptador pesa más (como subir el `lr`) | El adaptador pesa menos | α = 2·r (o α = r) |
| Módulos objetivo (`target_modules`) | En qué matrices se ponen adaptadores | Mejor calidad, más parámetros | Más barato | Atención (`q`, `k`, `v`, `o`); agregar el MLP si no alcanza |
| `lr` | Tamaño del paso | Aprende rápido, puede diverger | Lento | 1e-4–3e-4 (más alto que en el ajuste completo) |
| `lora_dropout` | Apaga al azar parte del adaptador (Lección 9.8) | Más regularización | Menos | 0.05–0.1 |

---
## 4. 📖 Cuantización: guardar cada número con menos bits

Normalmente cada peso ocupa **32 bits** (o 16). Cuantizar a **int8** significa guardarlo como un entero entre −127 y 127, más **un único factor de escala** compartido:

```
escala = (valor absoluto más grande) / 127
entero = redondear(peso / escala)            ← lo que se guarda (8 bits)
peso_aproximado = entero × escala            ← lo que se usa al calcular
```

Hagámoslo a mano con 5 pesos:

In [ ]:
pesos = np.array([0.12, -0.50, 0.33, 0.07, 2.80])     # ✏️ prueba quitando el 2.80

def cuantizar(w, bits=8):
    q_max = 2 ** (bits - 1) - 1                     # 127 para 8 bits, 7 para 4 bits
    escala = np.abs(w).max() / q_max
    enteros = np.round(w / escala).clip(-q_max, q_max)
    return enteros, enteros * escala

for bits in [8, 4]:
    enteros, aprox = cuantizar(pesos, bits)
    print(f"{bits} bits → enteros {enteros.astype(int)} → aproximación {np.round(aprox, 3)}")

### 🔮 Ejercicio 8 — El valor atípico
Con 4 bits, ¿qué le pasó al peso `0.07`? ¿Y al `0.12`? ¿Quién tiene la culpa? (Pista: mira cómo se calcula la escala.)

✏️ Luego quita el `2.80` de la lista y vuelve a ejecutar.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con 4 bits la escala es 2.80 / 7 = 0.4, así que solo hay "escalones" cada 0.4: el `0.07` y el `0.12` se redondean a **0** (¡desaparecen!) y el `0.33` se vuelve `0.4`. La culpa es del **valor atípico** `2.80`: estira la escala y deja sin resolución a los pesos pequeños, que son la mayoría. Con 8 bits los escalones son de 0.022 y el daño es pequeño.

Si quitas el 2.80, la escala pasa a 0.5 / 7 ≈ 0.07 y con 4 bits obtienes 0.143, −0.5, 0.357 y 0.071: mucho mejor. Los LLMs tienen justamente algunos pesos atípicos enormes, y por eso cuantizarlos bien es delicado.
</details>

### 📖 Por tensor vs. por canal
Una forma de reducir el daño de los atípicos es usar **una escala por cada columna** (por canal) en lugar de **una escala para toda la matriz** (por tensor). Probémoslo con una matriz real de GPT-2: la `c_attn` del primer bloque (768 × 2,304 = 1.8 millones de pesos).

In [ ]:
def cuantizar_tensor(w, bits=8, por_canal=False):
    q_max = 2 ** (bits - 1) - 1
    if por_canal:
        escala = w.abs().amax(dim=0, keepdim=True) / q_max     # una escala por columna
    else:
        escala = w.abs().max() / q_max                          # una escala para toda la matriz
    return torch.round(w / escala).clamp(-q_max, q_max) * escala

W_gpt = gpt2.transformer.h[0].attn.c_attn.weight.data
print(f"Peso típico (media de |w|): {W_gpt.abs().mean():.3f} · peso más grande: {W_gpt.abs().max():.2f}")
for bits in [8, 4]:
    for por_canal in [False, True]:
        error = (cuantizar_tensor(W_gpt, bits, por_canal) - W_gpt).abs().mean() / W_gpt.abs().mean()
        print(f"{bits} bits · {'por canal ' if por_canal else 'por tensor'} → error relativo medio {error:6.1%}")

### 🔮 Ejercicio 9 — ¿Se rompe GPT-2?
Ahora cuantizaremos **todas** las matrices de los 12 bloques de GPT-2 y mediremos su **perplejidad** (Lección 13.5) sobre un párrafo. Más baja = mejor.

Ordena de mejor a peor: 32 bits (original) · 8 bits por tensor · 8 bits por canal · 4 bits por tensor · 4 bits por canal. ¿Alguna configuración "rompe" el modelo?

✍️ *Mi orden:*

In [ ]:
texto = ("The history of the city goes back more than two thousand years. It was founded as a small fishing village "
         "on the banks of the river, and over the centuries it grew into an important center of trade. Merchants from distant "
         "lands came to sell spices, cloth and metal, and the markets became famous across the region. Today the old town "
         "is visited by millions of tourists every year, who come to see its narrow streets, its churches and its museums.")
ids = tok(texto, return_tensors="pt").input_ids
originales = {k: v.clone() for k, v in gpt2.state_dict().items()}    # copia de seguridad de los pesos

def perplejidad():
    with torch.no_grad():
        return math.exp(gpt2(ids, labels=ids).loss.item())

def cuantizar_gpt2(bits, por_canal):
    gpt2.load_state_dict(originales)                                 # partimos siempre del original
    with torch.no_grad():
        for capa in gpt2.transformer.h.modules():
            if type(capa).__name__ == "Conv1D":                       # las matrices de atención y MLP
                capa.weight.copy_(cuantizar_tensor(capa.weight, bits, por_canal))

print(f"32 bits (original)    → perplejidad {perplejidad():8.1f}")
for bits, por_canal in [(8, False), (8, True), (4, False), (4, True)]:   # ✏️ prueba 3 y 2 bits
    cuantizar_gpt2(bits, por_canal)
    print(f"{bits} bits {'por canal ' if por_canal else 'por tensor'}   → perplejidad {perplejidad():8.1f}")
_ = gpt2.load_state_dict(originales)                                 # dejamos el modelo como estaba

<details><summary><b>👀 Ver respuesta</b></summary>

**32 bits (≈12.2) ≈ 8 bits por canal (≈12.2) < 8 bits por tensor (≈14.5) < 4 bits por canal (≈17) ≪ 4 bits por tensor (≈2,560).**
- 8 bits por canal es prácticamente idéntico al original con **la cuarta parte de memoria**.
- 4 bits **por tensor** rompe el modelo: el error medio de los pesos era ≈59 % (los atípicos de nuevo) y la perplejidad se dispara a miles: escribe sin sentido.
- 4 bits **por canal** sobrevive con una pérdida moderada. Los métodos modernos (GPTQ, AWQ, NF4 de QLoRA) usan escalas por grupos pequeños y trucos extra para que 4 bits quede casi tan bien como 8.

✏️ Si pruebas 3 o 2 bits por canal, la perplejidad sube a cientos o miles: hay un límite.
</details>

### 📖 Hiperparámetros de la cuantización y QLoRA

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| Bits por peso | Precisión de cada número | Más fiel, más memoria | Menos memoria; por debajo de 4 bits suele romperse | 8 bits casi sin pérdida · 4 bits con buenos métodos |
| Granularidad de la escala | Cuántos pesos comparten escala | (tensor → canal → grupos de 64–128) más fino = menos error, un poco más de memoria | Más grueso: los atípicos arruinan todo | Por canal o por grupos de 64–128 |
| Qué se cuantiza | Capas afectadas | Más ahorro | Más calidad | Las matrices de los bloques; dejar en 16 bits la capa de salida y los embeddings |

**QLoRA** junta las dos ideas: el modelo base se guarda **congelado en 4 bits** y sobre él se entrenan adaptadores LoRA en 16 bits. Así se puede ajustar un modelo de 7B en una sola GPU de ~15 GB.

> 🧪 **Para Colab (con GPU)** — así se ve en la práctica con las librerías `peft` y `bitsandbytes` (no lo ejecutes aquí):
> ```python
> from transformers import AutoModelForCausalLM, BitsAndBytesConfig
> from peft import LoraConfig, get_peft_model
> modelo = AutoModelForCausalLM.from_pretrained("nombre-del-modelo",
>              quantization_config=BitsAndBytesConfig(load_in_4bit=True))      # base en 4 bits
> config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05)
> modelo = get_peft_model(modelo, config)
> modelo.print_trainable_parameters()    # ≈ 0.1 % entrenable
> ```

---
## 5. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El modelo con LoRA no mejora nada | `r` o `alpha` demasiado pequeños, `lr` bajo, o adaptadores en módulos que no importan | Subir `r`/`alpha`, `lr` ≈ 1e-4–3e-4, agregar más módulos |
| Al empezar a entrenar, el modelo ya da basura | `B` no se inicializó en ceros (el adaptador cambia el modelo desde el paso 0) | Revisar la inicialización |
| Aprende la tarea nueva pero "olvida" todo lo demás | Ajuste completo agresivo: **olvido catastrófico** | LoRA, `r` menor, menos épocas |
| El modelo cuantizado escribe sin sentido | Muy pocos bits o escala por tensor con atípicos | 8 bits, escala por canal/grupo, métodos como GPTQ/AWQ |
| "Con LoRA el modelo es más rápido al usarlo" | Falso: LoRA ahorra memoria **al entrenar**; al usarlo cuesta lo mismo (o un poco más) | Fusionar el adaptador: W ← W + (α/r)·B·A |
| Esperar que LoRA "meta" conocimiento nuevo masivo | LoRA ajusta comportamiento y estilo; el conocimiento grande viene del pre-entrenamiento | RAG para conocimiento (Lección 13.8) |

### 🔮 Ejercicio 10 — Fusionar el adaptador
Después de entrenar, se puede sumar el adaptador a la matriz original: `W_nueva = W + (α/r)·B·A`. ¿Cambian las predicciones? ¿Cuántos parámetros tiene la capa fusionada?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(0)
lora = CapaLoRA(copy.deepcopy(base), r=2, alpha=4)
entrenar(lora, X_tr, Y_tr)

fusionada = nn.Linear(d, d, bias=False)
with torch.no_grad():
    fusionada.weight.copy_(lora.base.weight + lora.escala * lora.B @ lora.A)   # W + (α/r)·B·A
    diferencia = (fusionada(X_prueba) - lora(X_prueba)).abs().max().item()
print(f"Diferencia máxima entre LoRA y la capa fusionada: {diferencia:.2e}")
print(f"Parámetros de la capa fusionada: {sum(p.numel() for p in fusionada.parameters())}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No cambian:** la diferencia máxima es ≈3·10⁻⁶, puro redondeo numérico. Y la capa fusionada tiene **4,096** parámetros, los mismos que la original. Una vez fusionado, el modelo es exactamente igual de rápido que el original: LoRA no tiene costo al usarlo. Si prefieres no fusionar, puedes intercambiar adaptadores al vuelo (uno por cliente) sobre el mismo modelo base.
</details>

---
## 6. 📋 Ficha: LoRA

| | LoRA (*Low-Rank Adaptation*) |
|---|---|
| **Tipo** | Técnica de ajuste fino eficiente (*PEFT, parameter-efficient fine-tuning*) |
| **Intuición** | No reescribir el libro: pegarle notas adhesivas |
| **Cómo aprende** | Congela W y entrena dos matrices flacas A y B; el cambio es (α/r)·B·A |
| **Hiperparámetros clave** | `r`, `alpha`, módulos objetivo, `lr` |
| **✅ Fortalezas** | Entrena ≈0.1–1 % de los parámetros · Adaptadores de pocos MB · Muchos adaptadores para un solo modelo base · Menos olvido · Combinable con cuantización (QLoRA) |
| **❌ Debilidades** | Puede quedarse corto si el cambio necesario es grande · Un hiperparámetro más (`r`) |
| **Úsalo cuando…** | Quieres adaptar estilo, formato o una tarea concreta con recursos modestos |
| **Evítalo cuando…** | Lo que te falta es **conocimiento** actualizado (usa RAG) o un simple *prompt* ya basta (Lección 13.8) |

## 7. 🧠 Resumen en 5 puntos
1. Ajustar un LLM completo con Adam cuesta ≈16 bytes por parámetro: un modelo de 7B no cabe en una GPU normal.
2. **LoRA** congela el modelo y aprende el cambio como **B·A**, dos matrices flacas de **rango r**: entrenas menos del 1 % de los parámetros.
3. Si `r` es menor que lo que necesita la tarea → subajuste; con pocos datos, LoRA generaliza **mejor** que el ajuste completo.
4. La **cuantización** guarda cada peso en 8 o 4 bits con un factor de escala; 8 bits casi no cambia nada, 4 bits necesita escalas por canal/grupo.
5. **QLoRA** = base cuantizada en 4 bits + adaptadores LoRA: ajustar modelos grandes en una sola GPU.

---
## 8. 🏋️ Ejercicios finales

### 8.1 Repaso rápido (opción múltiple)

**1.** Una capa de 1,000 × 1,000 con LoRA de `r=10`. ¿Cuántos parámetros entrenables tiene el adaptador?
- a) 1,000,000 · b) 20,000 · c) 10,000 · d) 100

**2.** ¿Por qué `B` se inicializa en ceros?
- a) Para ahorrar memoria · b) Para que al inicio el modelo se comporte exactamente como el original · c) Porque así entrena más rápido · d) Es un error

**3.** Cuantizas un modelo a 4 bits **por tensor** y escribe disparates. ¿Qué cambio tiene más sentido?
- a) Bajar a 2 bits · b) Usar escala por canal o por grupos · c) Subir `r` de LoRA · d) Entrenar más épocas

**4.** ¿Cuánta memoria ocupan los pesos de un modelo de 7B cuantizado a 4 bits (aprox.)?
- a) 28 GB · b) 14 GB · c) 3.5 GB · d) 0.5 GB

**5.** Tu empresa quiere que el asistente conozca las políticas internas, que cambian cada mes. ¿Qué es lo más adecuado?
- a) Ajuste completo cada mes · b) LoRA cada mes · c) RAG con los documentos actualizados · d) Cuantizar a 2 bits

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** r·(entrada + salida) = 10·(1,000 + 1,000) = 20,000: un 2 % de la matriz.
2. **b)** Con B = 0, B·A = 0: el adaptador empieza sin cambiar nada y el entrenamiento parte del modelo original.
3. **b)** Los atípicos arruinan la escala única; con escalas más finas el error baja muchísimo (lo viste en GPT-2).
4. **c)** 7×10⁹ pesos × 0.5 bytes ≈ 3.5 GB (más un poco por las escalas).
5. **c)** Para conocimiento que cambia, recuperar los documentos en el momento (Lección 13.8) es más barato, más actualizado y más verificable que reentrenar.
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** ¿Por qué tiene sentido que el **cambio** necesario para adaptar un modelo sea de bajo rango, aunque la matriz original no lo sea? Piensa en la diferencia entre "aprender a escribir" y "aprender a escribir con el estilo de tu empresa".

**B. Caso de análisis 💼** Una startup quiere ofrecer un asistente de redacción personalizado para **2,000 clientes**, cada uno con su estilo. Opciones: (1) un ajuste completo por cliente; (2) un adaptador LoRA por cliente sobre un único modelo base; (3) un solo modelo para todos con instrucciones en el *prompt*. Compara costos de almacenamiento y calidad esperada. ¿Qué harías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** El modelo base ya sabe casi todo lo necesario (gramática, vocabulario, razonamiento). Adaptarlo a un estilo es ajustar **unas pocas tendencias**: más formal, ciertas palabras, cierta estructura. Esos cambios ocurren en pocas "direcciones" del espacio de pesos, por eso un `r` pequeño basta. Si la tarea fuera totalmente nueva (otro idioma desconocido), el cambio sería más grande y haría falta un `r` mayor o un ajuste completo.

**B.** (1) es inviable: 2,000 copias completas (con un modelo de 7B en 16 bits, ≈14 GB cada una: ¡28 TB!). (2) Una sola copia del modelo base más 2,000 adaptadores de unos pocos MB a decenas de MB: barato, y existen sistemas que sirven muchos adaptadores a la vez. (3) Es lo más barato y conviene probarlo **primero**; si el estilo no queda bien solo con el *prompt*, entonces LoRA por cliente. Una buena respuesta empieza por lo simple y mide antes de complicar.
</details>

### 8.3 🧩 Práctica integradora: adapta y comprime

Vas a adaptar la capa a una tarea nueva **más difícil** (un cambio de rango 6), elegir `r`, y luego comprimir el resultado. Cada paso tiene su respuesta.

**Paso 1 ▶️ — La tarea nueva.** Ejecuta la celda: crea un cambio de rango 6 y 300 ejemplos de entrenamiento.

In [ ]:
torch.manual_seed(5)
cambio6 = (torch.randn(d, 6) @ torch.randn(6, d)) / 8
g = torch.Generator().manual_seed(2)
X6 = torch.randn(300, d, generator=g)
Y6 = X6 @ (W_vieja + cambio6).T + 0.05 * torch.randn(300, d, generator=g)
Y6_prueba = X_prueba @ (W_vieja + cambio6).T

def error_prueba6(modelo):
    with torch.no_grad():
        return ((modelo(X_prueba) - Y6_prueba) ** 2).mean().item()
print(f"Error de la capa original: {error_prueba6(base):.3f}")

**Paso 2 🔮 — Elige `r`.** ¿Desde qué `r` esperas que el error caiga casi a cero? Ejecuta y compruébalo.

In [ ]:
for r in [2, 4, 6, 8, 16]:   # ✏️
    torch.manual_seed(0)
    m = CapaLoRA(copy.deepcopy(base), r=r, alpha=2 * r)
    entrenar(m, X6, Y6)
    print(f"r={r:>2} → error en prueba {error_prueba6(m):.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El error cae casi a cero **desde `r=6`** (≈0.0001), justo el rango del cambio real. Con `r=2` (≈2.9) y `r=4` (≈1.0) se queda corto: subajuste. Subir a 8 o 16 ya no mejora nada (≈0.0002). Regla práctica: busca el `r` más pequeño a partir del cual el error de validación deja de mejorar.
</details>

**Paso 3 ✏️ — Fusiona y cuantiza.** Toma tu mejor adaptador (✏️ `R_ELEGIDO`), fusiónalo con la capa y cuantiza la matriz fusionada a 8 y a 4 bits **por canal**. ¿Cuánto empeora el error?

In [ ]:
R_ELEGIDO = 8   # ✏️
torch.manual_seed(0)
m = CapaLoRA(copy.deepcopy(base), r=R_ELEGIDO, alpha=2 * R_ELEGIDO)
entrenar(m, X6, Y6)
W_fusion = (m.base.weight + m.escala * m.B @ m.A).detach()

for bits in [32, 8, 4]:
    capa = nn.Linear(d, d, bias=False)
    with torch.no_grad():
        # nn.Linear guarda (salida, entrada): transponemos para escalar por canal de salida
        capa.weight.copy_(W_fusion if bits == 32 else cuantizar_tensor(W_fusion.T, bits, por_canal=True).T)
    print(f"{bits:>2} bits → error en prueba {error_prueba6(capa):.4f} · memoria de la matriz {d * d * bits / 8 / 1024:.1f} KB")

<details><summary><b>👀 Ver respuesta</b></summary>

- 32 bits: error ≈0.0002 · 16 KB.
- 8 bits por canal: error ≈**0.0004** (sigue siendo excelente) · **4 KB**, la cuarta parte.
- 4 bits por canal: error ≈**0.08**: cientos de veces peor que en 8 bits, aunque todavía mucho mejor que no adaptar (≈5.9) · 2 KB.

8 bits es casi gratis; 4 bits ya cobra un precio visible en una matriz tan pequeña. En LLMs grandes, con escalas por grupos y métodos más finos, 4 bits funciona sorprendentemente bien.
</details>

**Paso 4 ✋ — Tu recomendación.** Escribe en 3 frases qué configuración (r, bits) recomendarías y por qué, como si se lo explicaras a alguien de finanzas.

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"Recomiendo un adaptador LoRA de rango 8: entrena unas 1,000 cifras en lugar de 4,000 y resuelve la tarea nueva casi a la perfección, porque el cambio necesario era de rango 6. Al usarlo, fusionado y guardado en 8 bits, ocupa la cuarta parte de memoria sin pérdida apreciable. Bajar a 4 bits ahorraría la mitad otra vez, pero el error se multiplica varias veces, así que solo lo haría si la memoria fuera el cuello de botella."*
</details>

---
## 📚 Para profundizar
- 📄 **Artículo:** Hu et al. (2021), *LoRA: Low-Rank Adaptation of Large Language Models*.
- 📄 **Artículo:** Dettmers et al. (2023), *QLoRA: Efficient Finetuning of Quantized LLMs*.
- 📘 **Documentación:** Hugging Face `peft` (LoRA, QLoRA) y `bitsandbytes` (cuantización a 8 y 4 bits).
- 📺 **Video:** "LoRA explained (and a bit about precision and quantization)", DeepFindr (inglés, con subtítulos).

---
⬅️ **Anterior:** [13.6 Hiperparámetros de generación: las perillas de la creatividad](13.6-hiperparametros-de-generacion.ipynb) · ➡️ **Siguiente:** [13.8 Prompting, RAG y agentes: sacarle partido a un LLM sin reentrenarlo](13.8-prompting-rag-y-agentes.ipynb)